# Transfer Learning with PyTorch

## Install Packages

In [1]:
import torch
import torch.nn as nn
import torch.optim as opt

In [6]:
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [8]:
import  pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [10]:
from pathlib import Path
import os

In [7]:
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch torchvision version: {torchvision.__version__}")

PyTorch version: 2.12.0
PyTorch torchvision version: 0.27.0


In [9]:
device = "cpu"

## Get the data

In [20]:
data_dir = Path(".") / "data/pizza_steak_suchi"
train_dir = data_dir / "train"
test_dir = data_dir / "test"

we are going to use model from `torchvision.models` so we need to adapt the size of our tensors to the size of the model we are going to use.

Here we are going to use: [`EfficientNet`](https://arxiv.org/pdf/1905.11946)

### Manunal transformation

In [27]:
train_transform = transforms.Compose([
    transforms.Resize(size=(224,224)),
    transforms.ToTensor(),
    # all pretrained models as per the documentation expects this:
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.244, 0.255])
])

test_transform = transforms.Compose([
    transforms.Resize(size=(224,224)),
    transforms.ToTensor(),
    # all pretrained models as per the documentation expects this:
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.244, 0.255])
])

In [24]:
train_dataset = datasets.ImageFolder(root=train_dir, 
                                     transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, 
                                    transform=test_transform)

In [25]:
train_dataloader = DataLoader(dataset=train_dataset, 
                              batch_size=32, 
                              shuffle=True, 
                              num_workers=os.cpu_count()-4, 
                              pin_memory=True, 
                              prefetch_factor=2)

test_dataloader = DataLoader(dataset=test_dataset, 
                             batch_size=32, 
                             num_workers=os.cpu_count()-4, 
                             pin_memory=True, 
                             prefetch_factor=2)

### Automatic transform for EfficientNet

As of `torchvision`v0.13+ there is now support for automatic data transform creation based on the pretrained model we use

In [28]:
# Get the weights
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT #Default = best available weights

In [30]:
# Get the transformed used to get the pretrained weights
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [31]:
# Create the dataloaders using automatic transform
from modular.datasetup import create_dataloaders

In [32]:
train_dataloader_auto, test_dataloader_auto, class_names = create_dataloaders(train_dir=train_dir,
                                                                              test_dir=test_dir,
                                                                              transform=auto_transforms,
                                                                              batch_size=32)

Train data:
Dataset ImageFolder
    Number of datapoints: 225
    Root location: data/pizza_steak_suchi/train
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )
Test data:
Dataset ImageFolder
    Number of datapoints: 75
    Root location: data/pizza_steak_suchi/test
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )


In [33]:
class_names

['pizza', 'steak', 'sushi']

## Getting a pre-trained model

there are several ways to get an pre-trained model

1. PyTorch domain libraries
2. Libraries like `timm`(torch image models)
3. Hugging Face
4. Paper with code


To chose a model: Experiment...

three things to consider:

a. Speed - how fast does it run (where does it live and for what application like does it need to make a prediction live or not ?)

b. Size - how big is the model (impact the hardware available)

c. Performance - how well does it work on our problem

In [34]:
# setting up a pretrained model
model_effnet = torchvision.models.efficientnet_b0(weights=weights)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/maximecollet/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100.0%


In [35]:
feat_layer = model_effnet.features
avg_pool_layer = model_effnet.avgpool
classifier_layer = model_effnet.classifier